In [2]:
#初始化 BERT 模型需要做的第一件事是加载 Config 对象
from transformers import BertConfig, BertModel

# 初始化 Config 类
config = BertConfig()

# 从 Config 类初始化模型
model = BertModel(config)

这个模型是可以运行并得到结果的，但它会输出胡言乱语；它需要先进行训练才能正常使用。

In [3]:
#config 中包含许多用于构建模型的属性
print(config)

BertConfig {
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.57.3",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}



In [4]:
#加载已经训练过的 Transformers 模型使用 from_pretrained() 方法：

from transformers import BertModel

model = BertModel.from_pretrained("bert-base-cased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

In [5]:
#保存模型
model.save_pretrained("C:\\Users\\27729\\Desktop\\NLP-Learning\\outputs")

config.json——构建模型架构所需的属性，包含一些元数据，例如 checkpoint 的来源，以及你上次保存 checkpoint 时所使用的Transformers版本。
pytorch_model.safetensors 文件被称为 state dictionary（状态字典） ；它包含了你的模型的所有权重。
这两个文件是相辅相成的；配置文件是构建你的模型架构所必需的，而模型权重就是你的模型参数。

使用 Transformers 模型进行推理
Transformer 模型只能处理数字——由 tokenizer 转化后的数字。
可以将输入转换为适当的框架张量。

编码文本
Transformer 模型通过将输入转换为数字来处理文本。在这里，我们将详细了解文本被分词器处理时发生的情况。以通过一个简单的分词器来观察这种转换：

In [11]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

encoded_input = tokenizer("Hello, I'm a single sentence!")
print(encoded_input)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'input_ids': [101, 8667, 117, 146, 112, 182, 170, 1423, 5650, 106, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


input_ids：token的数值表示
token_type_ids：这些参数告诉模型输入的哪一部分是句子 A，哪一部分是句子 B
attention_mask：这指示哪些标记应该被关注，哪些不应该被关注

In [12]:
#我们可以解码输入ID以恢复原始文本：
tokenizer.decode(encoded_input["input_ids"])

"[CLS] Hello, I ' m a single sentence! [SEP]"

In [13]:
#当传递多个句子时，分词器会为每个句子返回一个列表，每个列表对应一个字典值。
#我们还可以让分词器直接从 PyTorch 返回张量：

encoded_input = tokenizer("How are you?", "I'm fine, thank you!", return_tensors="pt")
print(encoded_input)

{'input_ids': tensor([[ 101, 1731, 1132, 1128,  136,  102,  146,  112,  182, 2503,  117, 6243,
         1128,  106,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


这两个列表的长度不一样！数组和张量必须是矩形的，所以我们不能直接把这些列表转换成 PyTorch 张量（或 NumPy 数组）。分词器提供了一个选项来解决这个问题：填充。

In [14]:
#填充输入
#如果我们要求分词器对输入进行填充，
#它会通过向比最长句子短的句子添加特殊的填充标记，
#使所有句子的长度相同：

encoded_input = tokenizer(
    ["How are you?", "I'm fine, thank you!"], padding=True, return_tensors="pt"
)
print(encoded_input)

{'input_ids': tensor([[ 101, 1731, 1132, 1128,  136,  102,    0,    0,    0,    0],
        [ 101,  146,  112,  182, 2503,  117, 6243, 1128,  106,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


现在我们得到了矩形张量！填充标记已被编码为0的输入 ID，它们的注意力掩码值也为 0。这是因为这些填充标记不应该被模型分析：它们不属于实际句子的一部分。

In [17]:
#截断输入
#张量可能过大，模型无法处理。
#BERT 仅使用长度不超过 512 个 token 的序列进行预训练，
#因此无法处理更长的序列。如果序列长度超过模型的处理能力，
#则需要使用以下truncation参数对其进行截断：

encoded_input = tokenizer(
    "This is a very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very long sentence.",
    truncation=True,
    max_length=10,
    return_overflowing_tokens=True,
)
print(encoded_input["input_ids"])

[[101, 1188, 1110, 170, 1304, 1304, 1304, 1304, 1304, 102], [101, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 102], [101, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 102], [101, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 102], [101, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 102], [101, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 102], [101, 1304, 1304, 1304, 1304, 1263, 5650, 119, 102]]


In [18]:
#添加特殊标记
#特殊标记对于 BERT 及其衍生模型尤为重要。
# 这些标记用于更好地表示句子边界，
# 例如句子的开头（[CLS]）或句子之间的分隔符（[SEP]）。

encoded_input = tokenizer("How are you?")
print(encoded_input["input_ids"])
tokenizer.decode(encoded_input["input_ids"])

[101, 1731, 1132, 1128, 136, 102]


'[CLS] How are you? [SEP]'

In [ ]:
#假设我们有几个句子：
sequences = ["Hello!", "Cool.", "Nice!"]
encoded_sequences = [
    [101, 7592, 999, 102],
    [101, 4658, 1012, 102],
    [101, 3835, 999, 102],
]

这是一个编码序列列表：一个列表列表。张量只接受矩形（形状规则的的列表：每一列元素的数量都相同）。这个数组已经是矩形了，因此将其转换为张量很容易：

In [8]:
import torch
model_inputs = torch.tensor(encoded_sequences)

In [10]:
#将张量输入给模型非常简单 —— 我们只需调用模型并输入：
output = model(model_inputs)
print(output)

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[ 4.4496e-01,  4.8276e-01,  2.7797e-01,  ..., -5.4032e-02,
           3.9393e-01, -9.4770e-02],
         [ 2.4943e-01, -4.4093e-01,  8.1772e-01,  ..., -3.1917e-01,
           2.2992e-01, -4.1172e-02],
         [ 1.3668e-01,  2.2518e-01,  1.4502e-01,  ..., -4.6914e-02,
           2.8224e-01,  7.5566e-02],
         [ 1.1789e+00,  1.6738e-01, -1.8187e-01,  ...,  2.4671e-01,
           1.0441e+00, -6.1967e-03]],

        [[ 3.6436e-01,  3.2464e-02,  2.0258e-01,  ...,  6.0110e-02,
           3.2451e-01, -2.0996e-02],
         [ 7.1866e-01, -4.8725e-01,  5.1740e-01,  ..., -4.4012e-01,
           1.4553e-01, -3.7545e-02],
         [ 3.3223e-01, -2.3271e-01,  9.4877e-02,  ..., -2.5268e-01,
           3.2172e-01,  8.1114e-04],
         [ 1.2523e+00,  3.5754e-01, -5.1321e-02,  ..., -3.7840e-01,
           1.0526e+00, -5.6255e-01]],

        [[ 2.4042e-01,  1.4718e-01,  1.2110e-01,  ...,  7.6062e-02,
           3.3564e-01,  2